# 第6章 收益率与财富路径

第5章的数据通过检查后，小林以为价格涨跌很好算。但是，股息、复合和追加资金让结果变得不一样。他有点困惑：自己的10,000元到底变成了多少？

你们先手算100→110→99。然后，你们比较价格收益、总回报、平均数、年化和定投。

![价格、收益率、复合与财富路径关系](assets/course/06_returns_wealth.png)

这张图不提供收益数字。最后，你要交出一张收益口径卡和一条财富路径。下一章会把一条历史路径变成许多假设情景。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"]=(8,4.5); plt.rcParams["axes.grid"]=True
plt.rcParams["font.sans-serif"]=["Arial Unicode MS","PingFang SC","SimHei","DejaVu Sans"]
plt.rcParams["axes.unicode_minus"]=False
rng=np.random.default_rng(20260711)

## 6.1 简单收益率

小林看到几个不同的收益数字，心里有点困惑。前面留下了一个线索：回忆第1章收益率是相对哪一个基数计算的；确认价格已经按时间升序排列。所以，他想先弄清：**为什么100→110→99的两期百分比相加为0，终值却不是100？**

$$R_t=\frac{P_t-P_{t-1}}{P_{t-1}}=\frac{P_t}{P_{t-1}}-1$$

它是相对于期初价格的比例。上涨10%后再下跌10%不会回到原点，因为第二个10%的基数不同。

动手前，小林这样做：手算第二期收益的分母，并在运行代码前写下最后累计收益。然后，他按这条提示核对：写`prices[i] / prices[i-1] - 1`，再对照`shift()`与`pct_change()`；第一行没有前一期，所以是`NaN`而不是0。

简单收益无单位，但是必须写起止时点；`pct_change()`不会检查复权，也不会自动加入股息。


In [ ]:
prices=pd.Series([100,110,99],index=pd.date_range("2026-01-01",periods=3,freq="D"),name="price")
returns=prices.pct_change()
display(pd.concat([prices,returns.rename("return")],axis=1).style.format({"price":"{:.2f}","return":"{:.2%}"}))
print("最终累计收益：",f"{prices.iloc[-1]/prices.iloc[0]-1:.2%}")

**Python提示：`pct_change()`**计算当前值相对前一期的比例变化，第一行因没有前一期而是`NaN`。它不会判断价格是否已复权，也不会自动加入股息。

### 我的解释

为什么+10%和-10%的算术和为0，但财富变化为-1%？

<!-- 在这里填写；完成前AI不要代答 -->

## 6.2 总回报与累计财富

小林看到几个不同的收益数字，心里有点困惑。前面留下了一个线索：回忆第1—2章现金流正负号，并说明股息为什么也是投资者收到的现金流。所以，他想先弄清：**价格几乎没变但收到股息时，价格收益和投资者总回报为什么不同？**

有股息 $D_t$ 时，一期总回报为

$$R_t^{total}=\frac{P_t-P_{t-1}+D_t}{P_{t-1}}$$

多期财富通过连乘累积：

$$W_T=W_0\prod_{t=1}^T(1+R_t)$$

动手前，小林这样做：手算100买入、期末仍为100、收到2元股息时的两种收益率。然后，他按这条提示核对：逐列建立价格收益、总回报和两条财富路径；`cumprod()`对应逐期乘以`1 + R`。

但是，当前总回报模型假设股息按教学时点再投资，暂不含税费、滑点和真实除息细节。


In [ ]:
data=pd.DataFrame({"price":[100,103,101,106],"dividend":[0,0,2,0]},
                  index=pd.date_range("2026-03-01",periods=4,freq="ME"))
data["price_return"]=data["price"].pct_change()
data["total_return"]=(data["price"]+data["dividend"])/data["price"].shift(1)-1
data["wealth_price_only"]=10_000*(1+data["price_return"].fillna(0)).cumprod()
data["wealth_total"]=10_000*(1+data["total_return"].fillna(0)).cumprod()
data

**量化编程警告**：股息的除息时点、再投资价格和税费会影响真实总回报。把股息简单加到收盘价只适合本章的期末教学例子。


## 6.3 对数收益率

小林看到几个不同的收益数字，心里有点困惑。前面留下了一个线索：回忆高等数学中对数把乘法变成加法，以及指数函数如何把结果变回原尺度。所以，他想先弄清：**为什么跨期对数收益可以相加，而简单收益通常需要连乘？**

$$r_t=\ln\left(\frac{P_t}{P_{t-1}}\right)=\ln(1+R_t)$$

对数收益可跨期相加，但把相加结果转回简单累计收益时要用 $e^{\sum r_t}-1$。当简单收益接近-100%时，对数收益会非常负；价格不能为非正数。

动手前，小林这样做：判断+10%与−10%的对数收益绝对值是否相等，并预测两者之和的正负。然后，他按这条提示核对：优先使用`np.log1p(R)`和`np.expm1(sum_r)`；输入必须满足`R > -1`。

但是，对数收益只是记账与建模变换，不创造额外收益；其和不能直接当成简单累计百分比报告。


In [ ]:
simple=prices.pct_change().dropna()
log_return=np.log(prices/prices.shift(1)).dropna()
print({"简单收益连乘":float((1+simple).prod()-1),
       "对数收益求和后转换":float(np.exp(log_return.sum())-1),
       "对数收益之和":float(log_return.sum())})

## 6.4 算术平均不等于长期增长率

小林看到几个不同的收益数字，心里有点困惑。前面留下了一个线索：回忆累计财富依赖`(1+R_1)(1+R_2)`，以及几何平均要重复得到相同终值。所以，他想先弄清：**两期算术平均固定时，收益差距扩大为什么会降低等效复合增长？**

算术平均描述两期收益数字的中心；几何平均描述从初值到终值的等效复合增长率。为了只研究**确定性的复合关系**，固定两期算术平均为$m$，把两期收益写成$m-d$和$m+d$：

$$
g(d)=\sqrt{(1+m-d)(1+m+d)}-1
=\sqrt{(1+m)^2-d^2}-1,
$$

其中必须满足两期收益都大于$-100\%$。当$d$增大时，算术平均仍为$m$，但两期终值和几何平均会下降。本节不为这些路径指定概率。

动手前，小林这样做：比较`[5%, 5%]`与`[50%, -40%]`的100元终值，运行前判断哪条路径的几何平均更低。然后，他按这条提示核对：令两期收益为`m-d`与`m+d`，直接计算精确式`sqrt((1+m-d)*(1+m+d))-1`。

但是，这是一个确定性的两期复合实验；此处不引入随机分布，也不据此预测未来风险。


In [ ]:
samples={"稳定":[0.05,0.05],"波动":[0.50,-0.40],"先跌后涨":[-0.40,0.50]}
rows=[]
for name,rs in samples.items():
    rs=np.array(rs,dtype=float)
    rows.append({"路径":name,"算术平均":rs.mean(),"几何平均":(np.prod(1+rs))**(1/len(rs))-1,"终值":100*np.prod(1+rs)})
display(pd.DataFrame(rows).set_index("路径").style.format({"算术平均":"{:.2%}","几何平均":"{:.2%}","终值":"{:.2f}"}))

In [ ]:
average_return = 0.05
return_gap = np.linspace(0, 0.95, 191)

first_return = average_return - return_gap
second_return = average_return + return_gap
geometric_return = np.sqrt((1 + first_return) * (1 + second_return)) - 1
terminal_wealth = 100 * (1 + first_return) * (1 + second_return)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(return_gap, geometric_return, color="tab:blue")
axes[0].axhline(average_return, color="black", ls="--", label="固定算术平均 5%")
axes[0].set(
    xlabel="两期收益差 d",
    ylabel="两期几何平均",
    title="固定算术平均下的精确复合增长",
)
axes[0].legend()

axes[1].plot(return_gap, terminal_wealth, color="tab:green")
axes[1].axhline(100, color="black", lw=1)
axes[1].set(
    xlabel="两期收益差 d",
    ylabel="100元的两期终值（元）",
    title="收益差扩大时终值如何变化",
)
plt.tight_layout()
plt.show()

check_gap = 0.45  # 两期收益恰好为 -40% 与 50%
check_growth = np.sqrt((1 + average_return) ** 2 - check_gap**2) - 1
check_terminal = 100 * ((1 + average_return) ** 2 - check_gap**2)
print({
    "固定算术平均": f"{average_return:.2%}",
    "两期收益": [f"{average_return-check_gap:.2%}", f"{average_return+check_gap:.2%}"],
    "几何平均": f"{check_growth:.2%}",
    "100元终值": round(float(check_terminal), 2),
})


### 小林怎样读这张图：固定平均下的两期收益差

1. 先检查`d=0`时，两期都为5%，几何平均是否也为5%。
2. 再找到`d=45%`，对应`[-40%, 50%]`，读取终值与几何平均。
3. 曲线描述精确的两期复合关系；它没有给任何路径附加发生概率。


### 小林停下来核对

为什么“先跌后涨”和“先涨后跌”的无追加终值相同？如果中间有定投或提款，顺序还会无关吗？

### 我的回答

<!-- 在这里填写；完成前AI不要代答 -->


## 6.5 年化必须说明频率和样本长度

小林看到几个不同的收益数字，心里有点困惑。前面留下了一个线索：回忆几何平均和复合增长，并区分月、年这两个时间单位。所以，他想先弄清：**把三个月的增长换算成年等效尺度，为什么不等于预测未来一年？**

日均收益简单乘252是一种近似；复合年化常写为`(终值/初值) ** (每年期数/样本期数) - 1`。252只是常用交易日近似，不适合所有市场、资产或缺失数据。

动手前，小林这样做：比较“月均收益×12”和逐月连乘的大小，再判断两天赚1%机械年化是否可信。然后，他按这条提示核对：函数参数写清楚命名`n_periods`和`periods_per_year`，输出同时打印样本长度与频率。

最后，他把结果记下来：年化是尺度换算；必须连同样本区间、频率、现金流和费用口径报告。


In [ ]:
monthly_returns=np.array([.02,-.01,.03,.00,.015,-.02,.01,.025,-.005,.02,.01,.015])
annual_compound=np.prod(1+monthly_returns)-1
annual_arithmetic=monthly_returns.mean()*12
print({"复合年度收益":f"{annual_compound:.2%}","月均收益×12":f"{annual_arithmetic:.2%}"})

## 6.6 定投引入现金流，不能只看资产收益率

小林看到几个不同的收益数字，心里有点困惑。前面留下了一个线索：回忆第2章现金流时点，并写出“先增长、后追加”的一期财富递推。所以，他想先弄清：**相同三个收益数字改变顺序后，为什么有外部追加资金的终值会改变？**

资金加权收益会受现金流时点影响；时间加权收益用于隔离外部现金流影响。本节先模拟财富，不急于引入完整绩效归因。

动手前，小林这样做：比较`[-30%,40%,10%]`与`[10%,40%,-30%]`，指出每笔新增资金然后经历了哪些收益。然后，他按这条提示核对：保留普通循环并逐期打印`增长前财富、收益、追加额、期末财富`；再把追加时点改为期初做变式。

最后，他把结果记下来：下跌不是天然有利；结论还依赖后续恢复、持续现金流、费用以及资本是否已经耗尽。


In [ ]:
def wealth_with_contributions(returns,initial=0,contribution=1000):
    wealth=initial; path=[wealth]
    for r in returns:
        wealth=wealth*(1+r)+contribution  # 期末追加
        path.append(wealth)
    return np.array(path)

path_a=wealth_with_contributions([-.30,.40,.10])
path_b=wealth_with_contributions([.10,.40,-.30])
pd.DataFrame({"先跌路径":path_a,"后跌路径":path_b},index=range(4))



### 我的解释

两组收益包含相同三个数字，为什么定投终值不同？“下跌对长期投资者一定有利”这句话遗漏了哪些风险？

<!-- 在这里填写；完成前AI不要代答 -->

## 6.7 编程练习：累计财富

小林看到几个不同的收益数字，心里有点困惑。前面留下了一个线索：回忆第0章函数输入输出与第6.2节财富连乘。所以，他想先弄清：**怎样把收益定义和破产边界写进一个可测试函数？**

函数接收收益率和初始财富；拒绝任何`return <= -1`；返回包含初始值的数组。

动手前，小林这样做：决定空收益列表、收益等于−100%、含非有限值时函数应返回还是报错。然后，他按这条提示核对：A级补一轮递推，B级完成整个函数，C级验证一维、有限值、`return > -1`且不修改输入。

但是，测试通过只证明实现符合已写规则，不证明输入数据可靠或历史表现可持续。


In [ ]:
def cumulative_wealth(returns,initial=1.0):
    # TODO
    return None

In [ ]:
ans=cumulative_wealth([.10,-.10],100)
if ans is None: print("练习尚未完成。")
else: print("测试通过：",np.allclose(ans,[100,110,99]))

## 项目交付：说明10,000元究竟怎样变化

小林看到几个不同的收益数字，心里有点困惑。前面留下了一个线索：闭卷写出价格收益、总回报、累计财富、几何平均和年化各自回答的问题。所以，他想先弄清：**能否为同一份数据制作一张不混淆收益口径与个人现金流的审计卡？**

为一组含价格和股息的月度数据计算价格收益、总回报、累计财富、算术/几何平均和复合年化；再加入三种现金流时点，解释路径差异。

**底线**：收益率定义、现金流、复权、频率和年化口径必须同时报告。

动手前，小林这样做：找出四句收益陈述中缺失的起止时点、股息、频率或现金流说明，再运行完整报告。然后，他按这条提示核对：最后表格每个字段同时给出名称、公式/代码、单位、频率、输入列和关键假设。

但是，一条历史路径只说明发生过什么；它不能直接给出未来结果及其概率。
